# 02 — Drift Detection & Analysis
PULSE-OPS | Data drift analysis using Evidently AI.

Covers:
- Simulating covariate drift with PhysicsBasedDriftSimulator
- Running Evidently drift reports (data drift, data quality)
- Interpreting PSI and drift share metrics
- Triggering retraining based on drift thresholds
- Visualizing drift over time

In [ ]:
import sys
from pathlib import Path

repo_root = Path("__file__").resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings("ignore")

print("Environment ready")

## 1. Load and Prepare Reference Data

In [ ]:
from data.fetchers.uci_fetcher import UCIFetcher
from data.processors.feature_engineer import FeatureEngineer

fetcher = UCIFetcher()
adult_df = fetcher.fetch_adult_dataset()

engineer = FeatureEngineer()
df_clean = engineer.handle_missing(adult_df.copy())
df_clean = engineer.encode_categoricals(df_clean)

# Use only numeric features for drift
num_df = df_clean.select_dtypes(include=["number"]).fillna(0)

n = len(num_df)
reference = num_df.iloc[: n // 2].reset_index(drop=True)
production_stable = num_df.iloc[n // 2 :].reset_index(drop=True)

print(f"Reference:   {reference.shape}")
print(f"Production:  {production_stable.shape}")
print(f"Features:    {list(reference.columns)}")

## 2. Simulate Drift Scenarios

In [ ]:
from data.fetchers.synthetic_drifter import PhysicsBasedDriftSimulator

drifter = PhysicsBasedDriftSimulator()

# Sudden drift — large distribution shift
sudden_drift = drifter.simulate_sudden_drift(production_stable.copy(), shift_magnitude=4.0)

# Gradual drift — slow mean shift
gradual_drift = drifter.simulate_gradual_drift(production_stable.copy(), n_steps=10, max_magnitude=3.0)

# Seasonal drift
seasonal_drift = drifter.simulate_seasonal_drift(production_stable.copy(), amplitude=2.0)

print(f"Sudden drift — age mean shift: {production_stable['age'].mean():.2f} → {sudden_drift['age'].mean():.2f}")
print(f"Gradual drift — age mean shift: {production_stable['age'].mean():.2f} → {gradual_drift['age'].mean():.2f}")
print(f"Seasonal drift — age mean shift: {production_stable['age'].mean():.2f} → {seasonal_drift['age'].mean():.2f}")

In [ ]:
# Visualise the distribution shifts
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
col = "age"

for ax, drifted, title in zip(
    axes,
    [sudden_drift, gradual_drift, seasonal_drift],
    ["Sudden Drift", "Gradual Drift", "Seasonal Drift"]
):
    reference[col].hist(ax=ax, bins=30, alpha=0.6, color="steelblue", label="Reference")
    drifted[col].hist(ax=ax, bins=30, alpha=0.6, color="tomato", label="Drifted")
    ax.set_title(f"{title} — '{col}' Distribution")
    ax.set_xlabel(col)
    ax.legend()

plt.suptitle("Distribution Shift Visualisation", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Run Evidently Drift Detection

In [ ]:
from drift.evidently_detector import EvidentlyDriftDetector

detector = EvidentlyDriftDetector()

common_cols = list(set(reference.columns) & set(sudden_drift.columns))

# No-drift baseline
report_stable = detector.detect_data_drift(reference[common_cols], production_stable[common_cols])

# Sudden drift
report_sudden = detector.detect_data_drift(reference[common_cols], sudden_drift[common_cols])

# Gradual drift
report_gradual = detector.detect_data_drift(reference[common_cols], gradual_drift[common_cols])

print("\nDrift Detection Results:")
print(f"  Stable:   dataset_drift={report_stable.dataset_drift}, share={report_stable.drift_share:.3f}")
print(f"  Sudden:   dataset_drift={report_sudden.dataset_drift}, share={report_sudden.drift_share:.3f}")
print(f"  Gradual:  dataset_drift={report_gradual.dataset_drift}, share={report_gradual.drift_share:.3f}")

In [ ]:
# Full drift suite (all presets)
suite = detector.run_full_suite(reference[common_cols], sudden_drift[common_cols])

print("Full Suite Results:")
print(f"  Data drift share:   {suite.data_drift.drift_share:.3f}")
print(f"  Dataset drift:      {suite.data_drift.dataset_drift}")
if suite.data_quality:
    print(f"  Data quality score: {suite.data_quality}")

## 4. Per-Feature Drift Scores

In [ ]:
# Compare drift scores across features
if hasattr(suite.data_drift, "feature_drift_scores") and suite.data_drift.feature_drift_scores:
    feature_scores = suite.data_drift.feature_drift_scores
    score_series = pd.Series(feature_scores).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ["tomato" if v > 0.05 else "steelblue" for v in score_series.values]
    score_series.plot(kind="bar", ax=ax, color=colors)
    ax.axhline(0.05, color="red", linestyle="--", label="p-value threshold 0.05")
    ax.set_title("Per-Feature Drift p-values (Sudden Drift Scenario)")
    ax.set_ylabel("p-value")
    ax.set_xlabel("Feature")
    ax.legend()
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Per-feature scores not available in this drift report format.")

## 5. Retraining Trigger Evaluation

In [ ]:
from retraining.trigger import DriftBasedRetrigger

trigger = DriftBasedRetrigger()

for scenario, report in [
    ("Stable", report_stable),
    ("Gradual", report_gradual),
    ("Sudden", report_sudden),
]:
    # Create a minimal suite wrapper
    suite_wrapper = detector.run_full_suite(reference[common_cols], reference[common_cols])
    suite_wrapper.data_drift = report
    should = trigger.should_retrain(suite_wrapper)
    urgency = trigger.compute_urgency(suite_wrapper)
    print(f"  {scenario:10s}: retrain={should}, urgency={urgency}, drift_share={report.drift_share:.3f}")

## 6. Drift Reporter — Save JSON Report

In [ ]:
from drift.drift_reporter import DriftReporter
import json

reporter = DriftReporter()
report_path = reporter.save_json_report(suite)

with open(report_path) as f:
    saved = json.load(f)

print(f"Report saved to: {report_path}")
print(f"Report keys: {list(saved.keys())}")

## 7. HTML Report Generation

In [ ]:
import os

html_path = str(repo_root / "data" / "drift_report_notebook.html")
os.makedirs(os.path.dirname(html_path), exist_ok=True)

path = detector.generate_html_report(suite, html_path)
size_kb = os.path.getsize(path) / 1024
print(f"HTML report: {path} ({size_kb:.1f} KB)")